In [1]:
#load packages
import numpy as np
import xarray as xr
import math
import csv

import matplotlib.pyplot as plt
%matplotlib inline

import os
import pandas as pd
import cmocean
import matplotlib.gridspec as gridspec

from scipy.stats import norm
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, zoomed_inset_axes

# import gsw_xarray as gsw_xr # seawater calculations - might not need this one
import gsw as gsw
## mapping packages
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from pyproj import Transformer, Geod
from shapely.geometry import LineString, Point
from scipy.signal import savgol_filter
from scipy.interpolate import griddata

In [2]:
cd /g/data/jk72/deg581/seqom/analysis/notebooks

/g/data/jk72/deg581/seqom/analysis/notebooks


In [3]:
#define functions
def inpolygon(xq, yq, xv, yv):
    from matplotlib import path
    shape = xq.shape
    xq = xq.reshape(-1)
    yq = yq.reshape(-1)
    xv = xv.reshape(-1)
    yv = yv.reshape(-1)
    q = [(xq[i], yq[i]) for i in range(xq.shape[0])]
    p = path.Path([(xv[i], yv[i]) for i in range(xv.shape[0])])
    return p.contains_points(q).reshape(shape)

from xgcm import Grid

# map u,v to rho points
def ROMSmetricsAndGrid(ds):
    ds = ds.rename({'eta_u': 'eta_rho', 'xi_v': 'xi_rho', 'xi_psi': 'xi_u', 'eta_psi': 'eta_v'})

    coords={'X':{'center':'xi_rho', 'inner':'xi_u'}, 
        'Y':{'center':'eta_rho', 'inner':'eta_v'}, 
        'Z':{'center':'s_rho', 'outer':'s_w'}}

    grid = Grid(ds, coords=coords, periodic=[])

    print('making pm/pn metrics')
    ds['pm_v'] = grid.interp(ds.pm, 'Y')
    ds['pn_u'] = grid.interp(ds.pn, 'X')
    ds['pm_u'] = grid.interp(ds.pm, 'X')
    ds['pn_v'] = grid.interp(ds.pn, 'Y')
    ds['pm_psi'] = grid.interp(grid.interp(ds.pm, 'Y'),  'X') # at psi points (eta_v, xi_u) 
    ds['pn_psi'] = grid.interp(grid.interp(ds.pn, 'X'),  'Y') # at psi points (eta_v, xi_u)
    print('making dx/dy')
    ds['dx'] = 1/ds.pm
    ds['dx_u'] = 1/ds.pm_u
    ds['dx_v'] = 1/ds.pm_v
    ds['dx_psi'] = 1/ds.pm_psi

    ds['dy'] = 1/ds.pn
    ds['dy_u'] = 1/ds.pn_u
    ds['dy_v'] = 1/ds.pn_v
    ds['dy_psi'] = 1/ds.pn_psi

#     ds['dz'] = grid.diff(ds.z_w, 'Z', boundary='fill')
#     ds['dz_w'] = grid.diff(ds.z_rho, 'Z', boundary='fill')
#     ds['dz_u'] = grid.interp(ds.dz, 'X')
#     ds['dz_w_u'] = grid.interp(ds.dz_w, 'X')
#     ds['dz_v'] = grid.interp(ds.dz, 'Y')
#     ds['dz_w_v'] = grid.interp(ds.dz_w, 'Y')

    ds['dA'] = ds.dx * ds.dy

    metrics = {
        ('X',): ['dx', 'dx_u', 'dx_v', 'dx_psi'], # X distances
        ('Y',): ['dy', 'dy_u', 'dy_v', 'dy_psi'], # Y distances
        # ('Z',): ['dz', 'dz_u', 'dz_v', 'dz_w', 'dz_w_u', 'dz_w_v'], # Z distances
        ('X', 'Y'): ['dA'] # Areas
    }
    grid = Grid(ds, coords=coords, metrics=metrics, periodic=[])

    return ds,grid



def add_zeros_to_4(date):
    if date<10:
        to_add = '000'
    elif date>9 & date<100:
        to_add = '00'
    elif date>99 & date < 1000:
        to_add = '0'
    else: 
        to_add = ''
    return to_add

def generateFileList(FilePath,prefix,datelist):
    filelist=[FilePath+prefix+add_zeros_to_4(datelist[0])+str(datelist[0])+'.nc']
    for dates in datelist[1:]:
        filenameToAppend=FilePath+prefix+add_zeros_to_4(dates)+str(dates)+'.nc'
        filelist.append(filenameToAppend)
    return filelist



# Load Data

In [9]:
# load data file

grd = xr.open_dataset('/g/data/jk72/deg581/se-qld-setup/data/proc/seqld_1km_v1.7_grd.nc')

FilePath='/g/data/jk72/deg581/seqom/seqom_v1.7_2012_continue2/' #

prefix='roms_his_'
timeRange = [17,18]
datelist = np.array(range(timeRange[0],timeRange[1],1))


fl=generateFileList(FilePath,prefix,datelist)
print(fl)

# ds=loadOverlappedNetcdfFileList(filelist=fl,overlapDays=7)

ds = xr.open_mfdataset(fl,chunks = {'ocean_time':1}, data_vars='minimal', compat='override',coords='minimal',parallel='False',join='right')

print(ds.nbytes/1e9,'G')

ds = ds.drop_vars(['ubar_eastward','vbar_northward','w','rho','shflux','ssflux','sustr','svstr','temp','salt','u','v','u_eastward','v_northward'])
print(ds.nbytes/1e9,'G')
ds

ds = ds.assign_coords({"lon_rho": grd.lon_rho})
ds = ds.assign_coords({"lat_rho": grd.lat_rho})

weights_area = (1/ds.pm)*(1/ds.pn)
weights_area.name = "weights"

print('making vertical coordinates')
Zo_rho = (ds.hc * ds.s_rho + ds.Cs_r * ds.h) / (ds.hc + ds.h)
z_rho =  ( ds.h) * Zo_rho
Zo_w = (ds.hc * ds.s_w + ds.Cs_w * ds.h) / (ds.hc + ds.h)
z_w = Zo_w * ( + ds.h) 
    
ds.coords['z_w0'] = z_w.where(ds.mask_rho, 0).transpose('s_w', 'eta_rho', 'xi_rho')
ds.coords['z_rho0'] = z_rho.where(ds.mask_rho, 0).transpose('s_rho', 'eta_rho', 'xi_rho')

ds['dz'] = (('s_rho', 'eta_rho', 'xi_rho'),np.diff(ds.z_w0,axis=0))


ds, grid = ROMSmetricsAndGrid(ds)

# ds_17_again = ds

# ds.close()

['/g/data/jk72/deg581/seqom/seqom_v1.7_2012_continue2/roms_his_0017.nc']
39.617066672 G
0.226430192 G
making vertical coordinates
making pm/pn metrics
making dx/dy


In [10]:
ds.load()


<xarray.Dataset>
Dimensions:         (tracer: 2, boundary: 4, s_rho: 31, s_w: 32, Nuser: 1,
                     eta_rho: 720, xi_rho: 735, xi_u: 734, eta_v: 719,
                     ocean_time: 73)
Coordinates: (12/15)
  * s_rho           (s_rho) float64 -0.9839 -0.9516 ... -0.04839 -0.01613
  * s_w             (s_w) float64 -1.0 -0.9677 -0.9355 ... -0.06452 -0.03226 0.0
    x_rho           (eta_rho, xi_rho) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    y_rho           (eta_rho, xi_rho) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    x_u             (eta_rho, xi_u) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    y_u             (eta_rho, xi_u) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    ...              ...
    y_psi           (eta_v, xi_u) float64 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
  * ocean_time      (ocean_time) datetime64[ns] 2016-01-02 ... 2016-12-27
    lon_rho         (eta_rho, xi_rho) float64 151.5 151.5 151.5 ... 158.8 158.8
    lat_rho         (eta_rho, xi_rho) float64 -31.2 -31.2 ... -24.01 -24.01
    z_w0            (s_w, eta_rho, xi_rho) float64 0.0 0.0 0.0 ... 0.0 0.0 0.0
    z_rho0          (s_rho, eta_rho, xi_rho) float64 0.0 0.0 ... -1.115 -1.116
Dimensions without coordinates: tracer, boundary, Nuser, eta_rho, xi_rho, xi_u,
                                eta_v
Data variables: (12/105)
    ntimes          int32 1051200
    ndtfast         int32 20
    dt              float64 30.0
    dtfast          float64 1.5
    dstart          datetime64[ns] 2000-01-01
    nHIS            int32 14400
    ...              ...
    dx_psi          (eta_v, xi_u) float64 951.2 951.2 ... 1.016e+03 1.016e+03
    dy              (eta_rho, xi_rho) float64 1.112e+03 1.112e+03 ... 1.112e+03
    dy_u            (eta_rho, xi_u) float64 1.112e+03 1.112e+03 ... 1.112e+03
    dy_v            (eta_v, xi_rho) float64 1.112e+03 1.112e+03 ... 1.112e+03
    dy_psi          (eta_v, xi_u) float64 1.112e+03 1.112e+03 ... 1.112e+03
    dA              (eta_rho, xi_rho) float64 1.058e+06 1.058e+06 ... 1.129e+06
Attributes: (12/35)
    file:              roms_his_0017.nc
    format:            netCDF-3 64bit offset file
    Conventions:       CF-1.4, SGRID-0.3
    type:              ROMS/TOMS history file
    title:             South-east Queensland, 1/100 (900m) degree resolution
    var_info:          ROMS/External/varinfo.yaml
    ...                ...
    compiler_command:  /apps/openmpi/4.0.2/bin/mpif90
    compiler_flags:    -fp-model precise -heap-arrays -ip -O3 -traceback -che...
    tiling:            024x020
    history:           ROMS/TOMS, Version 4.2, Saturday - January 10, 2026 - ...
    ana_file:          ROMS/Functionals/ana_btflux.h
    CPP_options:       SEQLD, ANA_BSFLUX, ANA_BTFLUX, ASSUMED_SHAPE, AVERAGES...

## stats for 20 - 100m

In [30]:
log_gate = (ds.h<100) & (ds.h>20)
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').min(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').mean(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').max(dim='s_rho'))


<xarray.DataArray 'dz' (s_rho: 31)>
array([2.20634418, 2.18926233, 2.15134864, 2.09659572, 2.02898443,
       1.95224359, 1.86969841, 1.78419225, 1.6980642 , 1.61316566,
       1.53090157, 1.45228499, 1.37799661, 1.30844372, 1.24381492,
       1.18412872, 1.12927526, 1.07905102, 1.03318699, 0.99137099,
       0.95326491, 0.91851759, 0.88677417, 0.85768263, 0.83089786,
       0.80608411, 0.78291588, 0.76107783, 0.74026382, 0.72017545,
       0.70052008])
Coordinates:
  * s_rho    (s_rho) float64 -0.9839 -0.9516 -0.9194 ... -0.04839 -0.01613
<xarray.DataArray 'dz' ()>
array(0.70052008)
<xarray.DataArray 'dz' ()>
array(1.31866221)
<xarray.DataArray 'dz' ()>
array(2.20634418)


## stats for 100 - 1000m

In [31]:
log_gate = (ds.h>100) & (ds.h<1000)
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').min(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').mean(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').max(dim='s_rho'))
print('top 200m')
log_gate = (ds.h>100) & (ds.h<1000) & (ds.z_rho0>-200)
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').min(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').mean(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').max(dim='s_rho'))

<xarray.DataArray 'dz' (s_rho: 31)>
array([39.82831066, 39.39539322, 38.4345195 , 37.04687728, 35.33335601,
       33.38845818, 31.29645693, 29.12941345, 26.94660884, 24.79496454,
       22.71008717, 20.71765103, 18.8349077 , 17.07217927, 15.43424545,
       13.92157558, 12.53138527, 11.25851643, 10.09615152,  9.03637896,
        8.07062943,  7.19000319,  6.38550724,  5.64821943,  4.96939402,
        4.34052149,  3.75335255,  3.19989515,  2.67239098,  2.16327711,
        1.66513701])
Coordinates:
  * s_rho    (s_rho) float64 -0.9839 -0.9516 -0.9194 ... -0.04839 -0.01613
<xarray.DataArray 'dz' ()>
array(1.66513701)
<xarray.DataArray 'dz' ()>
array(17.3311537)
<xarray.DataArray 'dz' ()>
array(39.82831066)
top 200m
<xarray.DataArray 'dz' (s_rho: 31)>
array([ 9.50922603,  9.8608108 , 10.18640462, 10.4750815 , 10.68855716,
       10.7285473 , 11.30135645, 11.78729513, 13.7294757 , 13.96726136,
       14.1189252 , 14.25396188, 14.3905763 , 14.72169727, 15.33388686,
       13.92157558, 12.531

## stats for > 1000m

In [32]:
log_gate = (ds.h>1000)
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').min(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').mean(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').max(dim='s_rho'))
print('top 200m')
log_gate = (ds.h>1000) & (ds.z_rho0>-200)
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').min(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').mean(dim='s_rho'))
print(ds.dz.where(log_gate,drop=True).mean(dim='eta_rho').mean(dim='xi_rho').max(dim='s_rho'))

<xarray.DataArray 'dz' (s_rho: 31)>
array([238.61607135, 235.94453722, 230.0149841 , 221.45184193,
       210.87769966, 198.87573233, 185.96598952, 172.59316104,
       159.12307065, 145.84527053, 132.97948922, 120.68416288,
       109.06575106,  98.18795134,  88.08025931,  78.74557136,
        70.1667049 ,  62.31182942,  55.13887391,  48.5990159 ,
        42.63937413,  37.20502827,  32.24048263,  27.69067845,
        23.5016459 ,  19.62087256,  15.99745217,  12.58206575,
         9.32683679,   6.18509434,   3.11107103])
Coordinates:
  * s_rho    (s_rho) float64 -0.9839 -0.9516 -0.9194 ... -0.04839 -0.01613
<xarray.DataArray 'dz' ()>
array(3.11107103)
<xarray.DataArray 'dz' ()>
array(99.78608289)
<xarray.DataArray 'dz' ()>
array(238.61607135)
top 200m
<xarray.DataArray 'dz' (s_rho: 16)>
array([27.80351052, 27.3457627 , 27.28124576, 29.05338709, 30.83301159,
       32.39151522, 37.20502827, 32.24048263, 27.69067845, 23.5016459 ,
       19.62087256, 15.99745217, 12.58206575,  9.32683679,